# Step 01 — Build the Knowledge Graph from the Legacy MySQL Schema

**Goal:** Understand the source system before writing a single line of dbt.

We use a **synthetic e-commerce schema** (MySQL) as our legacy source.  
This schema simulates a real-world application database with a nightly ETL pipeline following the **medallion architecture**:

| Layer | Tables | Owned by |
|-------|--------|----------|
| Bronze (LZ) | `orders`, `customers`, `products` | Application — raw source data |
| Silver (Stage) | `slv_orders`, `slv_customers`, `slv_products` | ETL team — cleaned & typed, populated nightly |
| Gold (Marts) | `gold_fct_orders`, `gold_dim_customer` | ETL team — fact & dimension tables with business measures |

The target platform is **Databricks**, and we will use **dbt** to express the transformations.

The knowledge graph we build here answers:
- What tables exist and what are their columns?
- How are tables related (foreign keys and ETL data flows)?
- Which objects are sources, which are transformations, which are final outputs?

This graph becomes the **context layer** that feeds the migration agent in later notebooks.

```
legacy_schema/ecommerce_legacy.sql  (MySQL DDL + stored procedures)
        ↓  sqlglot  (parse DDL → AST)
  tables + columns + procedure dependencies
        ↓  NetworkX
  Knowledge Graph  (nodes = tables/procs/views, edges = FK + touches + depends_on)
        ↓
  Visualization + JSON export  →  used by later notebooks
```

**Stack:** `sqlglot` · `networkx` · `matplotlib` · `pandas`


## Step 1 — Install & import libraries

In [1]:
# %pip install --upgrade openai --quiet


In [ ]:
import os
import sqlglot
from sqlglot import exp
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import json
from pathlib import Path
from collections import defaultdict
from openai import OpenAI
from dotenv import load_dotenv
import networkx as nx
from typing import TypedDict, List, Dict, Any

load_dotenv()  # reads credentials from .env

print(f"sqlglot  : {sqlglot.__version__}")
print(f"networkx : {nx.__version__}")


sqlglot  : 30.12.0
networkx : 3.6.1


In [3]:
SCHEMA_FILE = Path("legacy_schema/ecommerce_legacy.sql")
raw_sql = SCHEMA_FILE.read_text(encoding="utf-8")

# Quick preview
lines = raw_sql.splitlines()
print(f"Total lines in file : {len(lines)}")
print(f"First 20 lines:\n")
print("\n".join(lines[:20]))

Total lines in file : 292
First 20 lines:

-- ============================================================
-- Synthetic legacy e-commerce schema (MySQL)
-- Represents a real application DB + nightly ETL pipeline.
--
-- Structure:
--   Layer 1 — Source tables  : orders, customers, products
--   Layer 2 — Silver tables  : slv_orders, slv_customers, slv_products
--   Layer 3 — Gold tables    : gold_fct_orders, gold_dim_customer
--   Reporting view           : v_segment_revenue
--   ETL procedures (x5)      : load_slv_* → load_gold_fct_* / load_gold_dim_*
-- ============================================================


-- ============================================================
-- LAYER 1: SOURCE TABLES  (live application database)
-- Owned by the application; ETL team reads but does not write.
-- Data arrives with real-world messiness in the values.
-- ============================================================

CREATE TABLE orders (


## Step 2 — Classify the schema with an LLM

Instead of hardcoding `SQL_INPUT`, an LLM reads the raw SQL file and classifies every statement automatically.

**Why LLM for this?**  
A real legacy `.sql` file mixes tables, views, procedures, and triggers in arbitrary order. Regex and heuristics break on `BEGIN...END` blocks, multi-line comments, and quoted identifiers. An LLM handles all of this in one pass — and can also reason about *which layer* a table belongs to from naming conventions and column signatures.

**Two outputs, two consumers:**

| Output | Type | Used by |
|--------|------|---------|
| `SQL_INPUT` | `dict` of SQL string lists | sqlglot parser (Steps 4–6) — unchanged |
| `TABLE_LAYERS` | `{table_name: layer}` | graph builder (Step 7) — sets `node_type` per table |

Keeping them separate means the parser is completely unaware of layers, and the graph builder is completely unaware of raw SQL strings.


### Step 2a — Classifier contract

Three constants define the full interface between the LLM and the rest of the notebook.  
Keeping them here — not inline in the prompt string or buried inside the graph builder — means **one place to update** if the schema evolves.

| Constant | Injected into | Used by | Purpose |
|---|---|---|---|
| `OBJECT_TYPES` | LLM prompt | parser (Steps 4–6) | Keys the LLM must produce; must match `SQL_INPUT` keys exactly — zero translation needed |
| `TABLE_LAYER_VOCAB` | LLM prompt | validator | Controlled vocabulary — prevents the LLM from inventing synonyms like `"staging"` or `"mart"` that the graph builder would silently not recognise |
| `LAYER_NODE_TYPE` | — | graph builder (Step 7) | Maps layer string → `node_type`; used only when creating graph nodes |

**Why all tables in one `"tables"` list, not split by layer?**  
The sqlglot parser (Step 4) extracts column names, data types, constraints, and foreign keys the same way for every table — it has no concept of layers. Splitting into `"source_tables"` / `"silver_tables"` / `"gold_tables"` would require changes to every parsing cell with zero benefit. Layer assignment happens exactly once: in Step 7 when the graph node is created.


In [4]:
# Keys the LLM must produce → must match SQL_INPUT keys exactly (no translation needed)
OBJECT_TYPES = ["tables", "views", "stored_procedures", "triggers", "functions"]

# Allowed layer values → injected into the prompt to constrain LLM output
TABLE_LAYER_VOCAB = ["source", "silver", "gold"]

# Layer → graph node_type → used only in Step 7 (graph builder), not by the parser
LAYER_NODE_TYPE = {
    "source": "source_table",   # → dbt source(), declared in sources.yml
    "silver": "silver_table",   # → dbt staging model, SELECT from source
    "gold":   "gold_table",     # → dbt fct_/dim_ model, SELECT from silver
}

print("Contract defined:")
print(f"  OBJECT_TYPES      : {OBJECT_TYPES}")
print(f"  TABLE_LAYER_VOCAB : {TABLE_LAYER_VOCAB}")
print(f"  LAYER_NODE_TYPE   : {LAYER_NODE_TYPE}")


Contract defined:
  OBJECT_TYPES      : ['tables', 'views', 'stored_procedures', 'triggers', 'functions']
  TABLE_LAYER_VOCAB : ['source', 'silver', 'gold']
  LAYER_NODE_TYPE   : {'source': 'source_table', 'silver': 'silver_table', 'gold': 'gold_table'}


In [5]:
import re as _re

ENDPOINT   = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
DEPLOYMENT = os.environ["AZURE_OPENAI_DEPLOYMENT"]
_api_key   = os.environ["AZURE_OPENAI_API_KEY"]

# AZURE_OPENAI_ENDPOINT points at a Microsoft Foundry *project* endpoint
# ("…/api/projects/<name>"). That path is used by the azure-ai-projects SDK
# for agents/evaluations — it is NOT a valid base for direct chat completions,
# which is why every api-version probe used to come back HTTP 400 BadRequest.
# Strip it down to the resource root, e.g.:
#   https://ai-models-for-migration.services.ai.azure.com/api/projects/proj-dev
#   -> https://ai-models-for-migration.services.ai.azure.com
RESOURCE_ROOT = _re.sub(r"/api/projects/.*$", "", ENDPOINT)

# The unified "v1" OpenAI-compatible endpoint needs no api-version query param
# and works whether the resource lives under *.services.ai.azure.com or
# *.openai.azure.com.
client = OpenAI(
    api_key  = _api_key,
    base_url = f"{RESOURCE_ROOT}/openai/v1/",
)

print(f"Resource root : {RESOURCE_ROOT}")
print(f"Client ready — deployment: {DEPLOYMENT}")


Resource root : https://ai-models-for-migration.services.ai.azure.com
Client ready — deployment: gpt-5


In [6]:
# Build the system prompt from the external template file.
# Placeholders <<object_types>> and <<layer_vocab>> are replaced at runtime from the
# contract constants above — so the file stays static while the values stay in one place.
_types_str  = ", ".join(f'"{t}"' for t in OBJECT_TYPES)
_layers_str = ", ".join(f'"{l}"' for l in TABLE_LAYER_VOCAB)

PROMPT_FILE = Path("skills/sql_classifier_prompt.md")
SYSTEM_PROMPT = (PROMPT_FILE.read_text(encoding="utf-8")
    .replace("<<object_types>>", _types_str)
    .replace("<<layer_vocab>>",  _layers_str))

print(f"System prompt loaded from {PROMPT_FILE}")
print(f"  Object types : {_types_str}")
print(f"  Layer vocab  : {_layers_str}")


System prompt loaded from skills\sql_classifier_prompt.md
  Object types : "tables", "views", "stored_procedures", "triggers", "functions"
  Layer vocab  : "source", "silver", "gold"


In [7]:
# Send the raw SQL to the LLM for classification.
# This is the only cell that makes a network call — re-run selectively when iterating on the prompt.
# gpt-5 is a reasoning model: it does NOT support `temperature` (or top_p, etc.) —
# use `max_completion_tokens` (not `max_tokens`) and `reasoning_effort` instead.
response = client.chat.completions.create(
    model                 = DEPLOYMENT,
    messages              = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": raw_sql},
    ],
    max_completion_tokens = 16000,
    reasoning_effort      = "low",
    response_format       = {"type": "json_object"},
)

if response.choices[0].finish_reason == "length":
    print("⚠ Response was truncated (finish_reason=length). Raise max_completion_tokens "
          "or lower reasoning_effort further.")
    print("usage:", response.usage)

# Strip markdown fences in case the model wraps the JSON (safety net)
raw_content = response.choices[0].message.content.strip()
if raw_content.startswith("```"):
    raw_content = raw_content.split("\n", 1)[-1].rsplit("```", 1)[0].strip()

result = json.loads(raw_content)
print(f"LLM returned {len(result)} top-level keys: {list(result.keys())}")


LLM returned 6 top-level keys: ['tables', 'views', 'stored_procedures', 'triggers', 'functions', 'table_layers']


In [8]:
# ── Unpack into the two separate consumers ───────────────────────────────────
# SQL_INPUT    → sqlglot parser (Steps 4–6); same dict structure as before
# TABLE_LAYERS → graph builder (Step 7); maps table_name → layer string
SQL_INPUT = {
    "dialect": "mysql",
    **{k: result.get(k, []) for k in OBJECT_TYPES},
}
TABLE_LAYERS = result.get("table_layers", {})

# ── Validate — fail fast before handing off to the parser ────────────────────
missing_keys   = [k for k in OBJECT_TYPES if k not in result]
unknown_layers = {v for v in TABLE_LAYERS.values() if v not in TABLE_LAYER_VOCAB}
assert not missing_keys,   f"LLM output missing required keys: {missing_keys}"
assert not unknown_layers, f"LLM used unknown layer values: {unknown_layers}"

# ── Summary ───────────────────────────────────────────────────────────────────
print("SQL_INPUT buckets:")
for k in OBJECT_TYPES:
    print(f"  {k:20s}: {len(SQL_INPUT[k])} statement(s)")
print()
print("TABLE_LAYERS:")
for name, layer in TABLE_LAYERS.items():
    print(f"  {name:30s} → {layer}")


SQL_INPUT buckets:
  tables              : 8 statement(s)
  views               : 1 statement(s)
  stored_procedures   : 5 statement(s)
  triggers            : 0 statement(s)
  functions           : 0 statement(s)

TABLE_LAYERS:
  orders                         → source
  customers                      → source
  products                       → source
  slv_orders                     → silver
  slv_customers                  → silver
  slv_products                   → silver
  gold_fct_orders                → gold
  gold_dim_customer              → gold


## Step 3 — SQL_INPUT: the parser's input contract

The LLM classifier above produced `SQL_INPUT` — a dict the sqlglot parser (Steps 4–6) reads exclusively.  
The parser never touches the raw file or `TABLE_LAYERS`; it only sees this structure:

```python
{
    "dialect": "mysql",
    "tables":            [...],   # CREATE TABLE statements (all layers combined)
    "views":             [...],   # CREATE VIEW statements
    "stored_procedures": [...],   # CREATE PROCEDURE statements
    "triggers":          [...],   # CREATE TRIGGER statements
    "functions":         [...],   # CREATE FUNCTION statements
}
```

**Why all tables in one list?**  
The parser extracts columns, types, constraints, and foreign keys the same way for every table — layer is irrelevant at parse time. Layer only matters later when we build the graph node and need to set `node_type`. That assignment is handled in Step 7 using `TABLE_LAYERS` + `LAYER_NODE_TYPE`.


## Step 4 — Parse the `tables` bucket with sqlglot

`sqlglot.parse()` turns a SQL string into an **Abstract Syntax Tree (AST)** — a tree of Python objects where every keyword, table name, column, and constraint becomes a node you can inspect.

Let's first look at what the AST looks like for our `actor` table, before extracting anything from it.

In [9]:
dialect = SQL_INPUT["dialect"]

# Parse just the first table to see what we get
first_table_sql = SQL_INPUT["tables"][0]
statements = sqlglot.parse(first_table_sql, dialect=dialect)
stmt = statements[0]

print("Top-level AST node type :", type(stmt).__name__)
print("stmt.this type          :", type(stmt.this).__name__)   # Schema
print("stmt.this.this type     :", type(stmt.this.this).__name__)  # Table
print()
print("sqlglot re-formats the SQL back for us (pretty=True):")
print(stmt.sql(pretty=True, dialect=dialect))

Top-level AST node type : Create
stmt.this type          : Schema
stmt.this.this type     : Table

sqlglot re-formats the SQL back for us (pretty=True):
CREATE TABLE orders (
  order_id INT UNSIGNED NOT NULL AUTO_INCREMENT,
  customer_id INT UNSIGNED NOT NULL,
  product_id INT UNSIGNED NOT NULL,
  quantity SMALLINT UNSIGNED NOT NULL DEFAULT 1,
  unit_price DECIMAL(10, 2) NOT NULL,
  order_date DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP(),
  status_code TINYINT NOT NULL DEFAULT 1, /* 1=pending 2=shipped 3=delivered 4=cancelled */
  created_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP(),
  updated_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP() ON UPDATE CURRENT_TIMESTAMP(),
  PRIMARY KEY (order_id),
  INDEX idx_orders_customer (customer_id),
  INDEX idx_orders_date (order_date),
  CONSTRAINT fk_orders_customer FOREIGN KEY (customer_id) REFERENCES customers (
    customer_id
  ),
  CONSTRAINT fk_orders_product FOREIGN KEY (product_id) REFERENCES products (
    product_id
  )
)
ENGI

### Step 4a — Extract table name and columns

Inside the AST the children of `stmt.this` (the `Schema` node) are the column definitions and constraints.  
We walk those to get: **table name**, **column name**, **data type**, and whether it is **nullable**.

In [10]:
parsed_tables = {}  # table_name -> {columns: [...]}

for sql in SQL_INPUT["tables"]:
    stmt = sqlglot.parse(sql, dialect=dialect)[0]

    table_name = stmt.this.this.name  # Create → Schema → Table → name

    columns = []
    for col_def in stmt.find_all(exp.ColumnDef):
        dtype = col_def.args.get("kind")
        nullable = not any(
            isinstance(c.args.get("this"), exp.NotNullColumnConstraint)
            for c in col_def.args.get("constraints", [])
        )
        columns.append({
            "name":     col_def.name,
            "type":     dtype.sql(dialect=dialect) if dtype else "UNKNOWN",
            "nullable": nullable,
        })

    parsed_tables[table_name] = {"columns": columns}

# Print what we extracted
for table, meta in parsed_tables.items():
    print(f"\nTable: {table}")
    for col in meta["columns"]:
        null_flag = "NULL" if col["nullable"] else "NOT NULL"
        print(f"  {col['name']:30s} {col['type']:30s} {null_flag}")


Table: orders
  order_id                       INT UNSIGNED                   NULL
  customer_id                    INT UNSIGNED                   NULL
  product_id                     INT UNSIGNED                   NULL
  quantity                       SMALLINT UNSIGNED              NULL
  unit_price                     DECIMAL(10, 2)                 NULL
  order_date                     DATETIME                       NULL
  status_code                    TINYINT                        NULL
  created_at                     TIMESTAMP                      NULL
  updated_at                     TIMESTAMP                      NULL

Table: customers
  customer_id                    INT UNSIGNED                   NULL
  first_name                     VARCHAR(100)                   NULL
  last_name                      VARCHAR(100)                   NULL
  email                          VARCHAR(200)                   NULL
  country_code                   CHAR(3)                        NULL
 

### Step 4b — Extract foreign keys

Foreign keys are what give us the **edges** in the knowledge graph.  
A FK says: column X in table A references column Y in table B — that is a directed relationship.

In the AST, FK constraints appear as `exp.ForeignKey` nodes alongside the column definitions.

In [11]:
foreign_keys = []  # list of {from_table, from_col, to_table, to_col}

for sql in SQL_INPUT["tables"]:
    stmt = sqlglot.parse(sql, dialect=dialect)[0]
    from_table = stmt.this.this.name

    for fk in stmt.find_all(exp.ForeignKey):
        from_col = fk.expressions[0].name      # local column

        ref = fk.args["reference"]
        # sqlglot parses "REFERENCES actor (actor_id)" as a Schema node:
        #   Schema.this       = Table("actor")
        #   Schema.expressions = [Identifier("actor_id")]
        if isinstance(ref.this, exp.Schema):
            to_table = ref.this.this.name
            to_col   = ref.this.expressions[0].name if ref.this.expressions else None
        else:
            # plain Table node (no explicit column list)
            to_table = ref.this.name
            to_col   = None

        foreign_keys.append({
            "from_table": from_table,
            "from_col":   from_col,
            "to_table":   to_table,
            "to_col":     to_col,
        })

print(f"Found {len(foreign_keys)} foreign key(s):\n")
for fk in foreign_keys:
    to = f"{fk['to_table']}.{fk['to_col']}" if fk["to_col"] else fk["to_table"]
    print(f"  {fk['from_table']}.{fk['from_col']}  →  {to}")

Found 2 foreign key(s):

  orders.customer_id  →  customers.customer_id
  orders.product_id  →  products.product_id


## Step 5 — Parse `stored_procedures`

Stored procedures are the **core of the ETL pipeline** — they define exactly how data moves between layers. Each procedure reads from one layer and writes to the next.

| Procedure | Reads from | Writes to | Layer transition |
|---|---|---|---|
| `load_slv_orders` | `orders` | `slv_orders` | source → silver |
| `load_slv_customers` | `customers` | `slv_customers` | source → silver |
| `load_slv_products` | `products` | `slv_products` | source → silver |
| `load_gold_fct_orders` | `slv_orders`, `slv_products` | `gold_fct_orders` | silver → gold |
| `load_gold_dim_customer` | `slv_customers`, `gold_fct_orders` | `gold_dim_customer` | silver → gold |

What we extract: the **direction** of each table reference — read (`FROM`/`JOIN`) vs write (`INTO`/`TRUNCATE`) — which becomes `reads_from` and `writes_to` edges in the graph.

**Why regex and not sqlglot AST here?**  
The AST gives a flat list of all referenced tables but no directional context. To determine direction from the AST you'd need to inspect each table's parent node (`exp.Insert`, `exp.TruncateTable`, etc.) — fragile across versions. Regex on the SQL keywords is simpler and more reliable for this specific task.


In [12]:
import re

# Words that can appear after FROM/JOIN/INTO etc. but are NOT table names
_SQL_KEYWORDS = {
    "EACH", "ROW", "BEGIN", "END", "SELECT", "INSERT",
    "UPDATE", "DELETE", "NULL", "SET", "VALUES",
}

def _proc_name_from_sql(sql):
    """Regex fallback to extract procedure/trigger/function name from raw SQL."""
    m = re.search(r"CREATE\s+(?:TRIGGER|PROCEDURE|FUNCTION)\s+(\w+)", sql, re.IGNORECASE)
    return m.group(1) if m else "unknown"

parsed_procs = {}  # proc_name -> {reads_from: [...], writes_to: [...]}

for sql in SQL_INPUT["stored_procedures"]:
    stmt = sqlglot.parse(sql, dialect=dialect)[0]

    # Always use regex for read/write separation.
    # sqlglot's AST (find_all(exp.Table)) lists all referenced tables but cannot tell
    # us whether a reference is a read (FROM/JOIN) or a write (INTO/UPDATE/TRUNCATE).
    proc_name  = _proc_name_from_sql(sql)
    if not proc_name or proc_name == "unknown":
        proc_name = (stmt.this.name if not isinstance(stmt, exp.Command) and stmt.this else "unknown")

    # FROM / JOIN → source tables (reads)
    read_hits  = re.findall(r"\b(?:FROM|JOIN)\s+(\w+)", sql, re.IGNORECASE)
    # INSERT INTO / UPDATE → target tables (writes)
    write_hits = re.findall(r"\b(?:INTO|UPDATE)\s+(\w+)", sql, re.IGNORECASE)
    # TRUNCATE TABLE → also a write
    trunc_hits = re.findall(r"\bTRUNCATE\s+(?:TABLE\s+)?(\w+)", sql, re.IGNORECASE)

    reads  = {t for t in read_hits  if t.upper() not in _SQL_KEYWORDS} - {proc_name}
    writes = {t for t in write_hits + trunc_hits if t.upper() not in _SQL_KEYWORDS} - {proc_name}

    parsed_procs[proc_name] = {
        "reads_from": sorted(reads),
        "writes_to":  sorted(writes),
    }

for proc, meta in parsed_procs.items():
    print(f"\nProcedure: {proc}")
    for tbl in meta["reads_from"]:
        print(f"  reads_from → {tbl}")
    for tbl in meta["writes_to"]:
        print(f"  writes_to  → {tbl}")



Procedure: load_slv_orders
  reads_from → orders
  writes_to  → slv_orders

Procedure: load_slv_customers
  reads_from → customers
  writes_to  → slv_customers

Procedure: load_slv_products
  reads_from → products
  writes_to  → slv_products

Procedure: load_gold_fct_orders
  reads_from → slv_orders
  reads_from → slv_products
  writes_to  → gold_fct_orders

Procedure: load_gold_dim_customer
  reads_from → gold_fct_orders
  reads_from → slv_customers
  writes_to  → gold_dim_customer


### Step 5a — Parse triggers

In [13]:
# --- Parse triggers ---
parsed_triggers = {}  # trigger_name -> {timing, event, on_table, reads_from, writes_to}

for sql in SQL_INPUT.get("triggers", []):
    name = _proc_name_from_sql(sql)

    m = re.search(r"(BEFORE|AFTER)\s+(INSERT|UPDATE|DELETE)\s+ON\s+(\w+)", sql, re.IGNORECASE)
    timing   = m.group(1).upper() if m else None
    event    = m.group(2).upper() if m else None
    on_table = m.group(3)         if m else None

    read_hits  = re.findall(r"\b(?:FROM|JOIN)\s+(\w+)", sql, re.IGNORECASE)
    write_hits = re.findall(r"\b(?:INTO|UPDATE)\s+(\w+)", sql, re.IGNORECASE)
    reads  = {t for t in read_hits  if t.upper() not in _SQL_KEYWORDS and t != on_table}
    writes = {t for t in write_hits if t.upper() not in _SQL_KEYWORDS and t != on_table}

    parsed_triggers[name] = {
        "timing":     timing,
        "event":      event,
        "on_table":   on_table,
        "reads_from": sorted(reads),
        "writes_to":  sorted(writes),
    }

for name, meta in parsed_triggers.items():
    print(f"\nTrigger: {name}  [{meta['timing']} {meta['event']} ON {meta['on_table']}]")
    for tbl in meta["reads_from"]:
        print(f"  reads_from → {tbl}")
    for tbl in meta["writes_to"]:
        print(f"  writes_to  → {tbl}")


### Step 5b — Parse functions

In [14]:
# --- Parse functions ---
parsed_functions = {}  # func_name -> {return_type, reads_from, writes_to}

for sql in SQL_INPUT.get("functions", []):
    stmt = sqlglot.parse(sql, dialect=dialect)[0]

    if isinstance(stmt, exp.Command):
        func_name   = _proc_name_from_sql(sql)
        ret_m       = re.search(r"\bRETURNS\s+([\w()]+(?:,\s*\d+\))?)", sql, re.IGNORECASE)
        return_type = ret_m.group(1) if ret_m else "UNKNOWN"
        read_hits   = re.findall(r"\b(?:FROM|JOIN)\s+(\w+)", sql, re.IGNORECASE)
        write_hits  = re.findall(r"\b(?:INTO|UPDATE)\s+(\w+)", sql, re.IGNORECASE)
        reads  = {t for t in read_hits  if t.upper() not in _SQL_KEYWORDS}
        writes = {t for t in write_hits if t.upper() not in _SQL_KEYWORDS}
    else:
        func_name   = (stmt.this.name if stmt.this else "") or _proc_name_from_sql(sql)
        returns     = stmt.args.get("returns")
        return_type = returns.sql(dialect=dialect) if returns else "UNKNOWN"
        reads  = {tbl.name for tbl in stmt.find_all(exp.Table) if tbl.name and tbl.name != func_name}
        writes = set()

    parsed_functions[func_name] = {
        "return_type": return_type,
        "reads_from":  sorted(reads),
        "writes_to":   sorted(writes),
    }

for name, meta in parsed_functions.items():
    print(f"\nFunction: {name}  → returns {meta['return_type']}")
    for tbl in meta["reads_from"]:
        print(f"  reads_from → {tbl}")
    for tbl in meta["writes_to"]:
        print(f"  writes_to  → {tbl}")


## Step 6 — Parse the `views` bucket

A view is a named SELECT query. What we want from it is:
- The **view name** (this becomes a node in the graph)
- The **tables it reads from** (these become dependency edges: view → table)

sqlglot's `find_all(exp.Table)` walks the entire SELECT body and returns every table reference, including those in JOINs and subqueries.


In [15]:
parsed_views = {}  # view_name -> {reads_from: [table, ...]}

for sql in SQL_INPUT["views"]:
    stmt = sqlglot.parse(sql, dialect=dialect)[0]

    # The view name sits at stmt.this (a Table node for CREATE VIEW)
    view_name = stmt.this.name

    # Collect every table referenced in the SELECT body
    # We exclude the view name itself (sqlglot sometimes includes it)
    referenced = set()
    for tbl in stmt.find_all(exp.Table):
        name = tbl.name
        if name and name != view_name:
            referenced.add(name)

    parsed_views[view_name] = {"reads_from": sorted(referenced)}

# Print results
for view, meta in parsed_views.items():
    print(f"\nView: {view}")
    for dep in meta["reads_from"]:
        print(f"  reads_from → {dep}")



View: v_segment_revenue
  reads_from → gold_dim_customer
  reads_from → gold_fct_orders


## Step 7 — Build the NetworkX knowledge graph

We now assemble all parsed objects into a **directed graph** (`DiGraph`). Every node is a schema object; every edge is a typed, directional relationship.

**Nodes**

| Object | `node_type` | Key attributes |
|--------|-------------|----------------|
| Source table | `source_table` | `columns` |
| Silver table | `silver_table` | `columns` |
| Gold table | `gold_table` | `columns` |
| View | `view` | `reads_from` |
| Stored procedure | `procedure` | `reads_from`, `writes_to` |
| Trigger | `trigger` | `timing`, `event`, `on_table`, `reads_from`, `writes_to` |
| Function | `function` | `return_type`, `reads_from`, `writes_to` |
| Column | `column` | `col_type`, `nullable`, `is_pk`, `is_unique` |

**Edges**

| Edge type | Source → Target | Meaning |
|-----------|-----------------|---------|
| `references` | table / column → table / column | FK constraint |
| `reads_from` | view / proc / trigger / function → table | Reads data from this table |
| `writes_to` | proc / trigger → table | Writes data to this table |
| `fires_on` | trigger → table | Trigger is bound to this table |
| `derived_from` | view column → source column | Column lineage through a view |
| `has_column` | table / view → column | Structural ownership |

A directed edge `A → B` means **A depends on or produces B**.


In [16]:
G = nx.DiGraph()

# --- Nodes: tables ---
for table_name, meta in parsed_tables.items():
    layer     = TABLE_LAYERS.get(table_name, "source")
    node_type = LAYER_NODE_TYPE.get(layer, "table")
    G.add_node(table_name, node_type=node_type, columns=meta["columns"])

# --- Nodes: views ---
for view_name, meta in parsed_views.items():
    G.add_node(view_name, node_type="view", reads_from=meta["reads_from"])

# --- Nodes: procedures ---
for proc_name, meta in parsed_procs.items():
    G.add_node(proc_name, node_type="procedure",
               reads_from=meta["reads_from"], writes_to=meta["writes_to"])

# --- Nodes: triggers ---
for trig_name, meta in parsed_triggers.items():
    G.add_node(trig_name, node_type="trigger",
               timing=meta["timing"], event=meta["event"], on_table=meta["on_table"],
               reads_from=meta["reads_from"], writes_to=meta["writes_to"])

# --- Nodes: functions ---
for func_name, meta in parsed_functions.items():
    G.add_node(func_name, node_type="function",
               return_type=meta["return_type"],
               reads_from=meta["reads_from"], writes_to=meta["writes_to"])

# --- Edges: foreign keys → REFERENCES ---
for fk in foreign_keys:
    G.add_edge(fk["from_table"], fk["to_table"],
               edge_type="references", from_col=fk["from_col"], to_col=fk["to_col"])

# --- Edges: view READS_FROM tables ---
for view_name, meta in parsed_views.items():
    for tbl in meta["reads_from"]:
        G.add_edge(view_name, tbl, edge_type="reads_from")

# --- Edges: procedure READS_FROM / WRITES_TO tables ---
for proc_name, meta in parsed_procs.items():
    for tbl in meta["reads_from"]:
        G.add_edge(proc_name, tbl, edge_type="reads_from")
    for tbl in meta["writes_to"]:
        G.add_edge(proc_name, tbl, edge_type="writes_to")

# --- Edges: trigger FIRES_ON its target table, READS_FROM / WRITES_TO others ---
for trig_name, meta in parsed_triggers.items():
    if meta["on_table"]:
        G.add_edge(trig_name, meta["on_table"], edge_type="fires_on")
    for tbl in meta["reads_from"]:
        G.add_edge(trig_name, tbl, edge_type="reads_from")
    for tbl in meta["writes_to"]:
        G.add_edge(trig_name, tbl, edge_type="writes_to")

# --- Edges: function READS_FROM / WRITES_TO tables ---
for func_name, meta in parsed_functions.items():
    for tbl in meta["reads_from"]:
        G.add_edge(func_name, tbl, edge_type="reads_from")
    for tbl in meta["writes_to"]:
        G.add_edge(func_name, tbl, edge_type="writes_to")

# Tag auto-created nodes (edge targets not in SQL_INPUT) as "referenced"
for node, attrs in G.nodes(data=True):
    if "node_type" not in attrs:
        G.nodes[node]["node_type"] = "referenced"

print(f"Nodes : {G.number_of_nodes()}")
print(f"Edges : {G.number_of_edges()}")
print()
print("Node list:")
for node, attrs in G.nodes(data=True):
    print(f"  [{attrs['node_type']:14s}]  {node}")
print()
print("Edge list:")
for src, dst, attrs in G.edges(data=True):
    print(f"  {src:30s} --[{attrs['edge_type']}]--> {dst}")


Nodes : 14
Edges : 16

Node list:
  [source_table  ]  orders
  [source_table  ]  customers
  [source_table  ]  products
  [silver_table  ]  slv_orders
  [silver_table  ]  slv_customers
  [silver_table  ]  slv_products
  [gold_table    ]  gold_fct_orders
  [gold_table    ]  gold_dim_customer
  [view          ]  v_segment_revenue
  [procedure     ]  load_slv_orders
  [procedure     ]  load_slv_customers
  [procedure     ]  load_slv_products
  [procedure     ]  load_gold_fct_orders
  [procedure     ]  load_gold_dim_customer

Edge list:
  orders                         --[references]--> customers
  orders                         --[references]--> products
  v_segment_revenue              --[reads_from]--> gold_dim_customer
  v_segment_revenue              --[reads_from]--> gold_fct_orders
  load_slv_orders                --[reads_from]--> orders
  load_slv_orders                --[writes_to]--> slv_orders
  load_slv_customers             --[reads_from]--> customers
  load_slv_customers    

## Step 8 — Column nodes and lineage

Every column becomes a first-class node in the graph (`node_type="column"`).

**Step 8 (this cell):** Create column nodes for all table columns + structural edges.  
**Step 8a:** Trace view column lineage and add `derived_from` edges.  
**Step 8b (TODO):** Same for stored procedures (`INSERT INTO ... SELECT`).


In [17]:
# Create column nodes for every table column + HAS_COLUMN edges
# View columns are added in Step 8a (they come from the lineage trace, not from parsed_tables)
for table_name, meta in parsed_tables.items():
    for col in meta["columns"]:
        col_node = f"{table_name}.{col['name']}"
        G.add_node(col_node, node_type="column", table=table_name,
                   col_name=col["name"], col_type=col["type"], nullable=col["nullable"])
        G.add_edge(table_name, col_node, edge_type="has_column")

# Column-level REFERENCES edges (FK constraint at column granularity)
for fk in foreign_keys:
    if not fk["to_col"]:
        continue
    src_col_node = f"{fk['from_table']}.{fk['from_col']}"
    dst_col_node = f"{fk['to_table']}.{fk['to_col']}"
    if src_col_node in G and dst_col_node in G:
        G.add_edge(src_col_node, dst_col_node, edge_type="references")

col_nodes     = [n for n, d in G.nodes(data=True) if d.get("node_type") == "column"]
col_ref_edges = [(u, v) for u, v, d in G.edges(data=True)
                 if d.get("edge_type") == "references"
                 and G.nodes[u].get("node_type") == "column"]

print(f"Column nodes created    : {len(col_nodes)}")
print(f"Column-level REFERENCES : {len(col_ref_edges)}")
for u, v in col_ref_edges:
    print(f"  {u:35s}  --[REFERENCES]-->  {v}")


Column nodes created    : 73
Column-level REFERENCES : 2
  orders.customer_id                   --[REFERENCES]-->  customers.customer_id
  orders.product_id                    --[REFERENCES]-->  products.product_id


### Step 8a — View column lineage

Trace each view output column back to its source table column using sqlglot's `lineage()` function, then add:
- Column nodes for view output columns (`has_column` from the view)
- `derived_from` edges: `view.col → source_table.col`


In [18]:
from sqlglot.lineage import lineage as col_lineage

column_lineage = []  # {from_col, to_col, view, out_col, src_table, src_col}

# ── Collect column lineage from views ─────────────────────────────────────────
for sql in SQL_INPUT["views"]:
    stmt = sqlglot.parse(sql, dialect=dialect)[0]
    view_name   = stmt.this.name
    select_expr = stmt.expression  # the SELECT node inside CREATE VIEW

    for col_expr in select_expr.expressions:
        out_col = col_expr.alias_or_name
        if not out_col:
            continue
        try:
            root = col_lineage(column=out_col, sql=select_expr, dialect=dialect)
            for node in root.walk():
                if node.downstream:
                    continue  # not a leaf, keep walking
                src_table = (
                    node.source.name
                    if isinstance(node.source, exp.Table)
                    else node.source_name
                )
                src_col = node.name.split(".")[-1]
                if src_table and src_table != view_name:
                    column_lineage.append({
                        "from_col":  f"{view_name}.{out_col}",
                        "to_col":    f"{src_table}.{src_col}",
                        "view":      view_name,
                        "out_col":   out_col,
                        "src_table": src_table,
                        "src_col":   src_col,
                    })
        except Exception as e:
            print(f"  Could not trace {view_name}.{out_col}: {e}")

print(f"Found {len(column_lineage)} column lineage link(s):\n")
for entry in column_lineage:
    print(f"  {entry['from_col']:35s}  derived_from  {entry['to_col']}")

# ── Add view column nodes + DERIVED_FROM edges ────────────────────────────────
for entry in column_lineage:
    from_node = entry["from_col"]
    to_node   = entry["to_col"]

    if from_node not in G:
        G.add_node(from_node, node_type="column",
                   table=entry["view"], col_name=entry["out_col"])
    if to_node not in G:
        G.add_node(to_node, node_type="column",
                   table=entry["src_table"], col_name=entry["src_col"])

    if entry["view"] in G:
        G.add_edge(entry["view"], from_node, edge_type="has_column")
    if entry["src_table"] in G:
        G.add_edge(entry["src_table"], to_node, edge_type="has_column")

    G.add_edge(from_node, to_node, edge_type="derived_from")

deriv_edges = [(u, v) for u, v, d in G.edges(data=True) if d.get("edge_type") == "derived_from"]
print(f"\nDERIVED_FROM edges added : {len(deriv_edges)}")


Found 6 column lineage link(s):

  v_segment_revenue.customer_segment   derived_from  gold_dim_customer.customer_segment
  v_segment_revenue.country_code       derived_from  gold_dim_customer.country_code
  v_segment_revenue.customer_count     derived_from  gold_dim_customer.customer_id
  v_segment_revenue.total_revenue      derived_from  gold_fct_orders.gross_revenue
  v_segment_revenue.avg_order_value    derived_from  gold_fct_orders.gross_revenue
  v_segment_revenue.total_margin       derived_from  gold_fct_orders.gross_margin

DERIVED_FROM edges added : 6


### Step 8b — Procedure column lineage (TODO)

All 5 stored procedures follow the same pattern: `TRUNCATE target; INSERT INTO target (...) SELECT ... FROM source`. sqlglot's `lineage()` works on any SELECT — so the same approach as Step 8a applies.

**Implementation plan:**
1. For each procedure SQL, extract the `SELECT ... FROM ...` body using regex
2. Run `col_lineage(column=out_col, sql=select_expr, dialect=dialect)` for each output column (matching the INSERT column list)
3. Append results to `column_lineage` — the Step 8a code that adds `derived_from` edges handles the rest unchanged

This would add edges like:
```
slv_customers.first_name  --[DERIVED_FROM]-->  customers.first_name
gold_fct_orders.gross_revenue  --[DERIVED_FROM]-->  slv_orders.unit_price   (computed)
```

See `TODO.md` for the full design notes.


## Step 9 — Enrich column nodes with constraint metadata

This step (plus Step 9a) fills out the full column profile the migration agent works from. The table below is the single reference for every attribute/label a `column` node can carry.

| Attribute added | Source in AST | Meaning |
|-----------------|---------------|---------|
| `col_type` | `ColumnDef.args["kind"]` (Step 4a) | Column's data type as declared in the DDL (e.g. `VARCHAR(100)`, `DECIMAL(10,2)`) |
| `nullable` | absence of `NotNullColumnConstraint` (Step 4a) | Whether the column allows `NULL` values |
| `is_pk` | table-level `PrimaryKey` node | Column is part of the primary key |
| `is_unique` | `UniqueColumnConstraint` | Column has a unique constraint (includes PK) |
| `auto_increment` | `AutoIncrementColumnConstraint` | Value is auto-generated on insert (MySQL-specific; maps to `GENERATED ALWAYS AS IDENTITY` or a sequence on target platforms) |
| `default` | `DefaultColumnConstraint` | Default value expression (some are MySQL-specific functions like `NOW()`) |
| `accepted_values` | inline `ENUM(...)`/`SET(...)` type params (Step 9a) | Allowed value list for MySQL types that embed their values in the type definition |


In [19]:
for sql in SQL_INPUT["tables"]:
    stmt = sqlglot.parse(sql, dialect=dialect)[0]
    table_name = stmt.this.this.name

    # --- PRIMARY KEY columns (table-level: PRIMARY KEY (col1, col2)) ---
    pk_cols = set()
    for pk in stmt.find_all(exp.PrimaryKey):
        for col in pk.expressions:
            pk_cols.add(col.name)

    # --- Column-level constraints ---
    col_constraints = {}   # col_name -> {auto_increment, default, is_unique}
    for col_def in stmt.find_all(exp.ColumnDef):
        info = {"auto_increment": False, "default": None, "is_unique": False}
        for constraint in col_def.args.get("constraints", []):
            c = constraint.args.get("this")
            if isinstance(c, exp.AutoIncrementColumnConstraint):
                info["auto_increment"] = True
            elif isinstance(c, exp.DefaultColumnConstraint):
                info["default"] = c.this.sql(dialect=dialect) if c.this else None
            elif isinstance(c, exp.UniqueColumnConstraint):
                info["is_unique"] = True
        col_constraints[col_def.name] = info

    # --- Update column nodes in the graph ---
    for col in parsed_tables[table_name]["columns"]:
        col_name  = col["name"]
        node_id   = f"{table_name}.{col_name}"
        extra     = col_constraints.get(col_name, {})

        if node_id in G:
            G.nodes[node_id]["is_pk"]          = col_name in pk_cols
            G.nodes[node_id]["auto_increment"] = extra.get("auto_increment", False)
            G.nodes[node_id]["default"]        = extra.get("default")
            G.nodes[node_id]["is_unique"]      = extra.get("is_unique", False) or col_name in pk_cols

# Print enriched column nodes
print("Enriched column nodes:\n")
for node, attrs in G.nodes(data=True):
    if attrs.get("node_type") != "column":
        continue
    flags = []
    if attrs.get("is_pk"):          flags.append("PK")
    if not attrs.get("nullable", True): flags.append("NOT NULL")
    if attrs.get("is_unique"):      flags.append("UNIQUE")
    if attrs.get("auto_increment"): flags.append("AUTO_INCREMENT")
    if attrs.get("default"):        flags.append(f"DEFAULT={attrs['default']}")
    flag_str = "  [" + ", ".join(flags) + "]" if flags else ""
    print(f"  {node:35s}{flag_str}")

Enriched column nodes:

  orders.order_id                      [PK, UNIQUE]
  orders.customer_id                 
  orders.product_id                  
  orders.quantity                    
  orders.unit_price                  
  orders.order_date                  
  orders.status_code                 
  orders.created_at                  
  orders.updated_at                  
  customers.customer_id                [PK, UNIQUE]
  customers.first_name               
  customers.last_name                
  customers.email                    
  customers.country_code             
  customers.is_active                
  customers.signup_date              
  customers.created_at               
  customers.updated_at               
  products.product_id                  [PK, UNIQUE]
  products.product_name              
  products.category                  
  products.cost_price                
  products.list_price                
  products.is_active                 
  products.created_at 

### Step 9a — Accepted values for inline-constrained column types

Some SQL dialects (MySQL, PostgreSQL) allow column types that embed their allowed values directly in the type definition — no separate `CHECK` constraint needed:

```sql
-- MySQL
order_status  ENUM('pending', 'shipped', 'delivered', 'cancelled')
feature_flags SET('Trailers', 'Commentaries', 'Deleted Scenes')
```

These type-level constraints are invisible to a generic column parser (Step 4a reports the type as `ENUM` but not the values). This step extracts the value list and stores it as `accepted_values` on the column node.

**Migration implication:** Neither Databricks nor Spark SQL supports `ENUM` or `SET`. The target column becomes a plain `STRING`, and the value constraint becomes an `accepted_values` test in the dbt schema YAML — or a `CHECK` constraint at the table level.


In [20]:
for sql in SQL_INPUT["tables"]:
    stmt = sqlglot.parse(sql, dialect=dialect)[0]
    table_name = stmt.this.this.name

    for col_def in stmt.find_all(exp.ColumnDef):
        dtype = col_def.args.get("kind")
        if not dtype:
            continue

        node_id = f"{table_name}.{col_def.name}"
        if node_id not in G:
            continue

        # dtype.this is a DataType.Type enum — .value gives us the string ("ENUM", "SET")
        type_name = dtype.this.value if hasattr(dtype.this, "value") else str(dtype.this)
        if type_name.upper() in ("ENUM", "SET"):
            accepted = [lit.name for lit in dtype.expressions
                        if isinstance(lit, exp.Literal)]
            G.nodes[node_id]["accepted_values"] = accepted
            G.nodes[node_id]["mysql_type"]      = type_name.upper()

# Print columns that have accepted_values
print("Columns with accepted_values:\n")
for node, attrs in G.nodes(data=True):
    if attrs.get("node_type") != "column":
        continue
    vals = attrs.get("accepted_values")
    if vals:
        mtype = attrs.get("mysql_type", "")
        print(f"  {node}")
        print(f"    mysql_type      : {mtype}")
        print(f"    accepted_values : {vals}")
        print()

Columns with accepted_values:



## Step 10 — Export to Neo4j

The knowledge graph is stored in **Neo4j** — a native graph database that maps directly onto the node/relationship model we built.

**Node labels:** `SOURCE_TABLE` · `SILVER_TABLE` · `GOLD_TABLE` · `VIEW` · `PROCEDURE` · `TRIGGER` · `FUNCTION` · `COLUMN` · `REFERENCED`  
**Relationship types:** `REFERENCES` · `READS_FROM` · `WRITES_TO` · `FIRES_ON` · `DERIVED_FROM` · `HAS_COLUMN`

Nodes and edges are batch-written using Cypher `UNWIND` — one query per label/relationship type.

Credentials come from `.env` (`NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`) — never hardcoded in the notebook.

Before writing, the DB is fully reset: **all existing constraints are dropped** (a previous run may have loaded a different dataset, e.g. Sakila, leaving stale label constraints behind) and all nodes/relationships are deleted, then fresh constraints are created for this graph's labels.

```cypher
// All procedures that write to a specific table (what populates it?)
MATCH (p:PROCEDURE)-[:WRITES_TO]->(t:SILVER_TABLE {node_id: 'slv_orders'})
RETURN p.node_id

// Full ETL chain: source → silver → gold
MATCH (proc:PROCEDURE)-[:READS_FROM]->(src:SOURCE_TABLE)
MATCH (proc)-[:WRITES_TO]->(slv:SILVER_TABLE)
RETURN proc.node_id, src.node_id, slv.node_id

// Column-level lineage through a view
MATCH (v:COLUMN)-[:DERIVED_FROM*]->(src:COLUMN)
RETURN v.node_id, src.node_id

// All FK relationships
MATCH (a)-[:REFERENCES]->(b)
WHERE NOT (a:COLUMN)
RETURN a.node_id, b.node_id
```

In [21]:
from neo4j import GraphDatabase

# NetworkX node_type → Neo4j label
NODE_LABEL = {
    "source_table": "SOURCE_TABLE",
    "silver_table": "SILVER_TABLE",
    "gold_table":   "GOLD_TABLE",
    "view":         "VIEW",
    "procedure":    "PROCEDURE",
    "trigger":      "TRIGGER",
    "function":     "FUNCTION",
    "column":       "COLUMN",
    "referenced":   "REFERENCED",
}

# NetworkX edge_type → Neo4j relationship type
REL_TYPE = {
    "references":   "REFERENCES",    # FK constraint: table/column → referenced table/column
    "reads_from":   "READS_FROM",    # view/proc/trigger/function reads from this table
    "writes_to":    "WRITES_TO",     # proc/trigger writes to this table
    "fires_on":     "FIRES_ON",      # trigger fires on this table
    "derived_from": "DERIVED_FROM",  # view column is derived from this source column
    "has_column":   "HAS_COLUMN",    # table/view owns this column node
}


In [22]:
NEO4J_URI      = os.getenv("NEO4J_URI", "neo4j://127.0.0.1:7687")
NEO4J_USER     = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
if not NEO4J_PASSWORD:
    raise ValueError("NEO4J_PASSWORD not set — add NEO4J_URI/NEO4J_USER/NEO4J_PASSWORD to .env")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print(f"Connected to Neo4j at {NEO4J_URI}")

Connected to Neo4j at neo4j://127.0.0.1:7687


In [23]:
# Full reset: drop ALL existing constraints first (a prior run against this Neo4j instance
# may have loaded a different dataset, e.g. Sakila, whose label constraints would otherwise linger),
# then wipe all data and recreate constraints for the current label set.
with driver.session() as session:
    old_constraints = [r["name"] for r in session.run("SHOW CONSTRAINTS YIELD name")]
    for name in old_constraints:
        session.run(f"DROP CONSTRAINT `{name}` IF EXISTS")

    session.run("MATCH (n) DETACH DELETE n")

    for label in NODE_LABEL.values():
        session.run(
            f"CREATE CONSTRAINT IF NOT EXISTS FOR (n:{label}) REQUIRE n.node_id IS UNIQUE"
        )

print(f"Dropped {len(old_constraints)} old constraint(s), wiped data, created {len(set(NODE_LABEL.values()))} fresh constraint(s)")

Dropped 5 old constraint(s), wiped data, created 9 fresh constraint(s)


In [24]:
# Batch-write nodes grouped by label (UNWIND sends one query per label, not one per node)
from collections import defaultdict

node_batches = defaultdict(list)
for node_id, attrs in G.nodes(data=True):
    label = NODE_LABEL.get(attrs.get("node_type", ""), "NODE")
    props = {"node_id": node_id}
    for k, v in attrs.items():
        props[k] = json.dumps(v) if isinstance(v, (list, dict)) else v
    node_batches[label].append(props)

with driver.session() as session:
    for label, rows in node_batches.items():
        session.run(
            f"UNWIND $rows AS row MERGE (n:{label} {{node_id: row.node_id}}) SET n += row",
            rows=rows
        )
        print(f"  {label:12s}: {len(rows)} nodes")

  SOURCE_TABLE: 3 nodes
  SILVER_TABLE: 3 nodes
  GOLD_TABLE  : 2 nodes
  VIEW        : 1 nodes
  PROCEDURE   : 5 nodes
  COLUMN      : 79 nodes


In [25]:
# Batch-write edges grouped by relationship type
edge_batches = defaultdict(list)
for src, dst, attrs in G.edges(data=True):
    rel_type = REL_TYPE.get(attrs.get("edge_type", ""), "RELATED")
    props = {k: v for k, v in attrs.items() if v is not None}
    edge_batches[rel_type].append({"src": src, "dst": dst, "props": props})

with driver.session() as session:
    for rel_type, rows in edge_batches.items():
        session.run(
            f"UNWIND $rows AS row "
            f"MATCH (a {{node_id: row.src}}), (b {{node_id: row.dst}}) "
            f"CREATE (a)-[r:{rel_type}]->(b) SET r += row.props",
            rows=rows
        )
        print(f"  {rel_type:15s}: {len(rows)} relationships")

# Also save NetworkX JSON as a lightweight backup for reloading G in later notebooks
OUTPUT_DIR = Path("knowledge_graph")
OUTPUT_DIR.mkdir(exist_ok=True)
nx_data = nx.node_link_data(G)
with open(OUTPUT_DIR / "kg_graph.json", "w") as f:
    json.dump(nx_data, f, default=str)
print(f"\nNetworkX JSON backup saved to {OUTPUT_DIR / 'kg_graph.json'}")

  REFERENCES     : 4 relationships
  HAS_COLUMN     : 79 relationships
  READS_FROM     : 9 relationships
  WRITES_TO      : 5 relationships
  DERIVED_FROM   : 6 relationships

NetworkX JSON backup saved to knowledge_graph\kg_graph.json


In [26]:
# Patch: add HAS_COLUMN edges in Neo4j (any table layer/VIEW -[:HAS_COLUMN]-> COLUMN)
# Uses MERGE so safe to re-run; redundant with the edge batch write above but kept as a safety net
with driver.session() as session:
    result = session.run(
        "MATCH (t), (c:COLUMN) "
        "WHERE (t:SOURCE_TABLE OR t:SILVER_TABLE OR t:GOLD_TABLE OR t:VIEW) AND c.table = t.node_id "
        "MERGE (t)-[r:HAS_COLUMN]->(c) "
        "RETURN count(r) AS created"
    )
    print(f"HAS_COLUMN relationships: {result.single()['created']}")

HAS_COLUMN relationships: 79


In [27]:
hc_edges = [(u,v) for u,v,d in G.edges(data=True) if d.get('edge_type')=='has_column']
col_nodes_all = [n for n,d in G.nodes(data=True) if d.get('node_type')=='column']
col_with_parent = {v for _,v,d in G.edges(data=True) if d.get('edge_type')=='has_column'}
orphans = [n for n in col_nodes_all if n not in col_with_parent]
print(f"HAS_COLUMN edges in G : {len(hc_edges)}")
print(f"Column nodes total    : {len(col_nodes_all)}")
print(f"Orphan columns        : {orphans}")

HAS_COLUMN edges in G : 79
Column nodes total    : 79
Orphan columns        : []


TO DO later: add evaluation steps here (check / LLM evaluation)